In [1]:
from pathlib import Path

import pandas as pd

# Notebook cwd is notebooks/; datasets live at repo root dataset/ (gitignored).
DATASET_ROOT = (Path.cwd().parent / "s3/dataset") if Path.cwd().name == "notebooks" else Path("dataset")

# df_d = pd.read_parquet(DATASET_ROOT / "sample_smoke_test/transactions")
# df_1k = pd.read_parquet(DATASET_ROOT / "sample_1000_users/transactions")
df_2k = pd.read_parquet(DATASET_ROOT / "sample_2000_users/features/transactions")

# df_d_t = pd.read_parquet(DATASET_ROOT / "dummy/transactions")
# df_d_u = pd.read_parquet(DATASET_ROOT / "dummy/customers")
# df_d_i = pd.read_parquet(DATASET_ROOT / "dummy/articles")
# len(df_d_t), len(df_d_u), len(df_d_i)

# df_2k['t_dat'].min(), df_2k['t_dat'].max()
df_2k.columns

Index(['customer_id', 'article_id', 't_dat', 'price', 'item_category',
       'item_color', 'item_pop_7d', 'item_pop_30d', 'item_pop_180d',
       'item_category_pop_30d', 'item_category_pop_180d',
       'item_pop_same_7d_last_year', 'first_sold_date',
       'days_since_first_sold', 'item_recent_to_last_180d_ratio',
       'item_category_recent_to_lifetime_ratio', 'item_seasonality_strength',
       'user_category_pref_1y_rank1', 'user_category_pref_1y_rank2',
       'user_category_pref_1y_rank3', 'user_color_pref_1y_rank1',
       'user_color_pref_1y_rank2', 'user_days_since_last_purchase',
       'user_purchase_count_30d', 'user_purchase_count_180d',
       'user_decayed_price_avg', 'user_decayed_price_std',
       'user_item_repurchase', 'user_item_decayed_repurchase',
       'user_item_decayed_interaction_ratio', 'candidate_price',
       'user_item_price_decayed_zscore', 'txn_month_sin', 'txn_month_cos',
       'year', 'month'],
      dtype='str')

In [2]:
df_2k.columns

Index(['customer_id', 't_dat', 'article_id', 'price', 'sales_channel_id',
       'year', 'month'],
      dtype='str')

In [12]:
DATASET_ROOT = (Path.cwd().parent / "s3/dataset/sample_2000_users/features") if Path.cwd().name == "notebooks" else Path("s3/dataset/sample_2000_users/features")
df_d_u = pd.read_parquet(DATASET_ROOT / "users").sort_values(by='last_purchase_date')
df_d_u.head()

,customer_id,last_purchase_date,user_purchase_count_30d,user_purchase_count_180d,user_days_since_last_purchase,user_decayed_price_avg,user_decayed_price_std,user_category_pref_1y_rank1,user_category_pref_1y_rank2,user_category_pref_1y_rank3,user_color_pref_1y_rank1,user_color_pref_1y_rank2,user_color_pref_1y_rank3,feature_cutoff
1451,b9d6d52f11bbd68bdc3273fcd1919e2a0fe4edec45c071...,2018-09-23,0,0,548.0,0.012695,0.004237,NaN,NaN,NaN,NaN,NaN,NaN,2020-03-24
1737,dcf5c11c4122e8b5b6dab57e6b471553b14454809e59df...,2018-09-25,0,0,546.0,0.042695,0.014946,NaN,NaN,NaN,NaN,NaN,NaN,2020-03-24
908,780f25cfc785190780f42b4e5ef4005d2f87ee5827a657...,2018-09-26,0,0,545.0,0.033881,0.000001,NaN,NaN,NaN,NaN,NaN,NaN,2020-03-24
128,0ed0ef4c6dfcc699cd5e17f5475b2ec2dcd51e23602257...,2018-09-26,0,0,545.0,0.032186,0.003390,NaN,NaN,NaN,NaN,NaN,NaN,2020-03-24
681,5aa03548ba9099604bc81c1260294547ad777df3baadbb...,2018-09-26,0,0,545.0,0.016932,0.000001,NaN,NaN,NaN,NaN,NaN,NaN,2020-03-24


(10, 5, 10)

In [8]:
df_d_t.head()

,customer_id,t_dat,article_id,price,sales_channel_id,year,month
0,8fdb3b94e9dcbd55aaa3015ce5571277a88a4b1fcb5c08...,2019-07-23,0718278002,0.008458,2,2019,7
1,aeb0430b6f1eb45079d047466435c30ff1394013eec26a...,2019-07-08,0613147003,0.016932,2,2019,7
2,aeb0430b6f1eb45079d047466435c30ff1394013eec26a...,2019-07-24,0619884001,0.012203,1,2019,7
3,8fdb3b94e9dcbd55aaa3015ce5571277a88a4b1fcb5c08...,2019-07-23,0775313001,0.016932,2,2019,7
4,aeb0430b6f1eb45079d047466435c30ff1394013eec26a...,2019-08-20,0732311003,0.012695,1,2019,8


In [7]:
df_d_t.groupby('article_id').size()

article_id
0613147003    1
0619884001    1
0688537001    1
0718278002    1
0732311003    1
0772773002    1
0772902002    1
0775313001    1
0785709002    1
0804661001    1
dtype: int64

In [ ]:
----

In [ ]:
import json
import os
import shutil
from datetime import datetime, timezone
from functools import reduce
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

In [ ]:
if IS_GLUE:
    # AWS Glue PySpark notebook: SparkSession already exists as `spark`.
    pass
else:
    import subprocess
    from pyspark import SparkContext

    # IDE kernels often miss shell JAVA_HOME (see java-pyspark-local-setup.md).
    if not os.environ.get("JAVA_HOME"):
        for java_home_cmd in (
            ["/usr/libexec/java_home", "-v", "17"],
            ["/usr/libexec/java_home", "-v", "1.8"],
            ["/usr/libexec/java_home"],
        ):
            try:
                os.environ["JAVA_HOME"] = subprocess.check_output(
                    java_home_cmd, text=True, stderr=subprocess.DEVNULL
                ).strip()
                break
            except (subprocess.CalledProcessError, FileNotFoundError):
                continue

    def _reset_stale_spark() -> None:
        """Drop Python-side Spark singletons when the JVM gateway is dead."""
        SparkSession._instantiatedSession = None
        SparkContext._active_spark_context = None
        SparkContext._gateway = None
        SparkContext._jvm = None

    # Re-run safe: getOrCreate() reuses a dead JVM → ConnectionRefusedError.
    try:
        active = SparkSession.getActiveSession()
    except AssertionError:
        active = None
    if active is not None:
        try:
            active.sparkContext._jsc.sc().version()
        except Exception:
            _reset_stale_spark()
    elif SparkSession._instantiatedSession is not None or SparkContext._active_spark_context is not None:
        _reset_stale_spark()

    # Local driver: enough memory for full H&M CSVs on a laptop.
    spark = (
        SparkSession.builder.appName("stratified-user-sampling")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.shuffle.partitions", "8")
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")

In [ ]:
# number of rows in hive parquet dataset/sample_2000_users/transactions
import pyarrow.parquet as pq

parquet_path = DATASET_ROOT / "sample_2000_users/transactions"
table = pq.read_table(parquet_path)
num_rows = table.num_rows
print(f"Number of rows in {parquet_path}: {num_rows}")

